In [388]:
import numpy as np
import pandas as pd
import yfinance as yf

In [389]:
sector_dict = {

    "Banking": [
        "HDFCBANK.NS",
        "ICICIBANK.NS",
        "KOTAKBANK.NS",
        "SBIN.NS",
        "^NSEI"
    ],

    "IT": [
        "TCS.NS",
        "INFY.NS",
        "WIPRO.NS",
        "HCLTECH.NS",
        "^NSEI"
    ],

    "Energy": [
        "RELIANCE.NS",
        "ONGC.NS",
        "BPCL.NS",
        "^NSEI"
    ],

    "Auto": [
        "MARUTI.NS",
        "M&M.NS",
        "^NSEI"
    ],

    "FMCG": [
        "HINDUNILVR.NS",
        "ITC.NS",
        "NESTLEIND.NS",
        "^NSEI"
    ]
}

In [390]:
def download_data(tickers):
    
    data = yf.download(
        tickers,
        start="2012-01-01",
        end="2025-01-01",
        auto_adjust=False
    )
    
    data.columns = [
        f"{ticker}_{feature}"
        for feature, ticker in data.columns
    ]
    
    data = data.drop(
        columns=[
            col for col in data.columns
            if "Adj Close" in col
        ],
        errors="ignore"
    )
    
    data = data.dropna(how="all")
    
    return data

In [391]:
def select_core_columns(data):
    
    selected_cols = [
        col for col in data.columns
        if (
            "Close" in col
            or "Volume" in col
        )
    ]
    
    filtered_data = data[selected_cols]
    
    return filtered_data

In [392]:
def create_returns(data):
    
    core_data = select_core_columns(data)
    
    returns = core_data.pct_change()
    
    return returns

In [393]:
def create_momentum_features(data,windows=[3, 5, 10, 20]):
    
    core_data = select_core_columns(data)
    
    momentum_features = []
    
    for window in windows:
        
        momentum = (core_data.pct_change(window))
        
        momentum.columns = [
            f"{col}_mom_{window}"
            for col in momentum.columns
        ]
        
        momentum_features.append(momentum)
    
    momentum_df = pd.concat(momentum_features,axis=1)
    
    return momentum_df

In [394]:
def create_volatility_features(returns,windows=[5, 20]):
    
    volatility_features = []
    
    for window in windows:
        
        volatility = returns.rolling(window).std()
        
        volatility.columns = [
            f"{col}_vol_{window}"
            for col in volatility.columns
        ]
        
        volatility_features.append(volatility)
    
    volatility_df = pd.concat(
        volatility_features,
        axis=1
    )
    
    return volatility_df

In [395]:
def create_intraday_returns(data,tickers):
    
    intraday_returns = pd.DataFrame()
    
    for ticker in tickers:
        
        intraday_returns[
            f"{ticker}_intraday_return"
        ] = (
            (
                data[f"{ticker}_Close"]
                - data[f"{ticker}_Open"]
            )
            / data[f"{ticker}_Open"]
        )
    
    return intraday_returns

In [396]:
def create_hl_range(data,tickers):
    
    hl_range = pd.DataFrame()
    
    for ticker in tickers:
        
        hl_range[
            f"{ticker}_hl_range"
        ] = (
            (
                data[f"{ticker}_High"]
                - data[f"{ticker}_Low"]
            )
            / data[f"{ticker}_Close"]
        )
    
    return hl_range

In [397]:
def create_ma_distance_features(data,tickers,windows=[10, 20, 50]):
    
    ma_features = []
    
    for window in windows:
        
        ma_distance = pd.DataFrame()
        
        for ticker in tickers:
            
            ma = (
                data[f"{ticker}_Close"]
                .rolling(window)
                .mean()
            )
            
            ma_distance[
                f"{ticker}_ma_dist_{window}"
            ] = (
                (
                    data[f"{ticker}_Close"]
                    - ma
                )
                / ma
            )
        
        ma_features.append(ma_distance)
    
    ma_df = pd.concat(
        ma_features,
        axis=1
    )
    
    return ma_df

In [398]:
def create_relative_volume_features(data,tickers,window=20):
    
    relative_volume = pd.DataFrame()
    
    for ticker in tickers:
        
        if ticker == "^NSEI":
            continue
        
        volume_ma = (
            data[f"{ticker}_Volume"]
            .rolling(window)
            .mean()
        )
        
        relative_volume[
            f"{ticker}_rel_volume_{window}"
        ] = (
            data[f"{ticker}_Volume"]
            / volume_ma
        )
    
    return relative_volume

In [399]:
def create_rolling_mean_return_features(returns,windows=[5, 20]):
    
    rolling_mean_features = []
    
    for window in windows:
        
        rolling_mean = (
            returns
            .rolling(window)
            .mean()
        )
        
        rolling_mean.columns = [
            f"{col}_mean_ret_{window}"
            for col in rolling_mean.columns
        ]
        
        rolling_mean_features.append(
            rolling_mean
        )
    
    rolling_mean_df = pd.concat(
        rolling_mean_features,
        axis=1
    )
    
    return rolling_mean_df

In [400]:
def get_sector_tickers(sector_name,sector_dict):
    return sector_dict[sector_name]

In [401]:
def prepare_ticker_universe(target_ticker,sector_name,sector_dict):
    
    tickers = sector_dict[sector_name].copy()
    
    if target_ticker not in tickers:
        tickers.insert(0, target_ticker)
    
    tickers = list(set(tickers))
    
    return tickers

In [402]:
def build_feature_matrix(target_ticker,sector_name,sector_dict):
    
    
    tickers = prepare_ticker_universe(target_ticker,sector_name,sector_dict)
    
    
    data = download_data(tickers)
    
    
    returns = create_returns(data)
    
    
    momentum = create_momentum_features(data)
    
    
    volatility = create_volatility_features(returns)
    
    
    # intraday = create_intraday_returns(data,tickers)
    
    
    # hl_range = create_hl_range(data,tickers)
    
    
    ma_distance = create_ma_distance_features(data,tickers)
    
    
    # relative_volume = (create_relative_volume_features(data,tickers))
    
     
    rolling_mean = (create_rolling_mean_return_features(returns))
    
    
    all_features = pd.concat(
        [
            returns,
            momentum,
            volatility,
            ma_distance,
            rolling_mean
        ],
        axis=1)
    
    
    bad_cols = [
        col for col in all_features.columns
        if "^NSEI_Volume" in col
        or "^NSEI_rel_volume" in col
    ]

    raw_cols = [
    col for col in all_features.columns
    if col.endswith("_Close")
    or col.endswith("_Volume")
]
    

    all_features = all_features.drop(columns=raw_cols,errors="ignore")

    all_features = all_features.drop(
        columns=bad_cols,
        errors="ignore"
    )
    
    
    all_features = all_features.dropna()
    
    return all_features

In [403]:
features = build_feature_matrix("HDFCBANK.NS","Banking",sector_dict)

[*********************100%***********************]  5 of 5 completed


In [404]:
features.shape

(2428, 87)

In [405]:
features.isnull().sum().sum()

np.int64(0)

In [406]:
len(features.columns)

87

In [407]:
features.columns.tolist()

['HDFCBANK.NS_Close_mom_3',
 'ICICIBANK.NS_Close_mom_3',
 'KOTAKBANK.NS_Close_mom_3',
 'SBIN.NS_Close_mom_3',
 '^NSEI_Close_mom_3',
 'HDFCBANK.NS_Volume_mom_3',
 'ICICIBANK.NS_Volume_mom_3',
 'KOTAKBANK.NS_Volume_mom_3',
 'SBIN.NS_Volume_mom_3',
 'HDFCBANK.NS_Close_mom_5',
 'ICICIBANK.NS_Close_mom_5',
 'KOTAKBANK.NS_Close_mom_5',
 'SBIN.NS_Close_mom_5',
 '^NSEI_Close_mom_5',
 'HDFCBANK.NS_Volume_mom_5',
 'ICICIBANK.NS_Volume_mom_5',
 'KOTAKBANK.NS_Volume_mom_5',
 'SBIN.NS_Volume_mom_5',
 'HDFCBANK.NS_Close_mom_10',
 'ICICIBANK.NS_Close_mom_10',
 'KOTAKBANK.NS_Close_mom_10',
 'SBIN.NS_Close_mom_10',
 '^NSEI_Close_mom_10',
 'HDFCBANK.NS_Volume_mom_10',
 'ICICIBANK.NS_Volume_mom_10',
 'KOTAKBANK.NS_Volume_mom_10',
 'SBIN.NS_Volume_mom_10',
 'HDFCBANK.NS_Close_mom_20',
 'ICICIBANK.NS_Close_mom_20',
 'KOTAKBANK.NS_Close_mom_20',
 'SBIN.NS_Close_mom_20',
 '^NSEI_Close_mom_20',
 'HDFCBANK.NS_Volume_mom_20',
 'ICICIBANK.NS_Volume_mom_20',
 'KOTAKBANK.NS_Volume_mom_20',
 'SBIN.NS_Volume_mom_20'

In [408]:
def create_future_returns(data,target_ticker,horizon=5):
    future_returns = (
        data[f"{target_ticker}_Close"].pct_change(horizon).shift(-horizon)
    )


    future_returns = future_returns.dropna()

    return future_returns

In [409]:
def create_labels(future_returns,quantile=0.7):
    thresold = future_returns.quantile(quantile)

    labels = (
        future_returns > thresold
    ).astype(int)


    return labels

In [410]:
df = download_data("TCS.NS")

[*********************100%***********************]  1 of 1 completed


In [411]:
fut=create_future_returns(df,"TCS.NS")

In [412]:
create_labels(fut).value_counts()

TCS.NS_Close
0    2239
1     960
Name: count, dtype: int64

In [413]:
def prepare_dataset(target_ticker,sector_name,sector_dict,horizons=5,quantiles=0.7):
    
    features = build_feature_matrix(target_ticker,sector_name,sector_dict)

    tickers = prepare_ticker_universe(target_ticker,sector_name,sector_dict)

    data = download_data(tickers)

    future_returns = create_future_returns(data,target_ticker,horizon=horizons)

    label = create_labels(future_returns,quantiles)

    data = features.copy()

    data['Target'] = label

    data = data.dropna()

    return data 

In [414]:
dataset = prepare_dataset(
    "RELIANCE.NS",
    "Energy",
    sector_dict
)

[*********************100%***********************]  4 of 4 completed
[*********************100%***********************]  4 of 4 completed


In [415]:
dataset['Target'].value_counts()

Target
0.0    1689
1.0     734
Name: count, dtype: int64

In [416]:
dataset.head()

,BPCL.NS_Close_mom_3,ONGC.NS_Close_mom_3,RELIANCE.NS_Close_mom_3,^NSEI_Close_mom_3,BPCL.NS_Volume_mom_3,ONGC.NS_Volume_mom_3,RELIANCE.NS_Volume_mom_3,BPCL.NS_Close_mom_5,ONGC.NS_Close_mom_5,RELIANCE.NS_Close_mom_5,...,ONGC.NS_Volume_mean_ret_5,RELIANCE.NS_Volume_mean_ret_5,BPCL.NS_Close_mean_ret_20,ONGC.NS_Close_mean_ret_20,RELIANCE.NS_Close_mean_ret_20,^NSEI_Close_mean_ret_20,BPCL.NS_Volume_mean_ret_20,ONGC.NS_Volume_mean_ret_20,RELIANCE.NS_Volume_mean_ret_20,Target
2012-03-15,0.006636,0.032055,-0.002379,0.003909,0.029255,-0.106748,0.346782,0.012209,0.027245,0.045747,...,0.147856,-0.040383,0.005608,0.002042,-0.002914,-0.000243,0.168324,0.106895,0.109781,0.0
2012-03-16,-0.004496,-0.050426,-0.058052,-0.020554,1.039789,2.050906,0.561611,-0.000075,-0.034818,-0.001874,...,0.436727,0.085771,0.004895,-0.000865,-0.003718,-0.001895,0.225072,0.224219,0.057322,0.0
2012-03-19,-0.023619,-0.071830,-0.072928,-0.037858,-0.454860,0.055049,-0.093350,0.003846,-0.018008,-0.054411,...,0.437266,0.038339,0.003433,-0.001324,-0.003388,-0.002376,0.144857,0.224172,0.044911,0.0
2012-03-20,0.026821,-0.049730,-0.045315,-0.019636,1.015178,-0.020240,-0.483895,0.027051,-0.053034,-0.072443,...,0.343338,-0.045089,0.005210,-0.001379,-0.003381,-0.002591,0.195315,0.155000,0.012896,0.0
2012-03-21,0.046895,-0.005493,-0.005567,0.008848,0.294314,-0.675359,-0.435339,0.020172,-0.075574,-0.057029,...,0.372537,0.008067,0.006366,-0.003401,-0.004437,-0.002122,0.212110,0.148817,0.030114,0.0


In [417]:
X = dataset.drop(columns=['Target'])
Y = dataset['Target']

In [418]:
from sklearn.model_selection import train_test_split

x_train,x_temp,y_train,y_temp = train_test_split(X,Y,test_size=0.3,shuffle=False)

x_val,x_test,y_val,y_test = train_test_split(x_temp,y_temp,test_size=2/3,shuffle=False)

In [419]:
print(x_train.shape)
print(x_val.shape)
print(x_test.shape)

(1696, 68)
(242, 68)
(485, 68)


In [420]:
print(x_train.index.min(), x_train.index.max())

print(x_val.index.min(), x_val.index.max())

print(x_test.index.min(), x_test.index.max())

2012-03-15 00:00:00 2022-01-11 00:00:00
2022-01-12 00:00:00 2023-01-02 00:00:00
2023-01-03 00:00:00 2024-12-23 00:00:00


In [421]:
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix)

In [422]:
scaler = StandardScaler()

In [423]:
x_train_scaled = scaler.fit_transform(x_train)

x_val_scaled = scaler.fit_transform(x_val)

x_test_scaled = scaler.fit_transform(x_test)

In [424]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')

model.fit(x_train_scaled,y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [425]:
val_pred = model.predict(x_val_scaled)

In [426]:
w = list(x_train.columns)

In [427]:
w[0]

'BPCL.NS_Close_mom_3'

In [428]:
z = model.coef_

In [429]:
coeff_dict = {}
for i in range(0,x_train.shape[1]):
    coeff_dict[w[i]] = (z[0][i])

In [430]:
coef_df = pd.DataFrame(
    coeff_dict.items(),
    columns=["Feature", "Coefficient"]
)

In [431]:
coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()

In [432]:
coef_df = coef_df.sort_values(
    by="Abs_Coefficient",
    ascending=False
)

In [433]:
coef_df["Ticker"] = coef_df["Feature"].str.split("_").str[0]

In [434]:
coef_df.groupby("Ticker")["Abs_Coefficient"].sum().sort_values(ascending=False)

Ticker
BPCL.NS        4.466379
RELIANCE.NS    4.071339
ONGC.NS        3.871593
^NSEI          3.082401
Name: Abs_Coefficient, dtype: float64

In [435]:
def get_family(feature):

    if "mom" in feature:
        return "Momentum"

    elif "vol_" in feature:
        return "Volatility"

    elif "ma_dist" in feature:
        return "MA_Distance"

    elif "rel_volume" in feature:
        return "Relative_Volume"

    elif "mean_ret" in feature:
        return "Mean_Return"

    elif "intraday_return" in feature:
        return "Intraday"

    elif "hl_range" in feature:
        return "HL_Range"

    elif "Volume" in feature:
        return "Raw_Volume"

    elif "Close" in feature:
        return "Raw_Close"

    else:
        return "Other"

In [436]:
coef_df["Family"] = coef_df["Feature"].apply(get_family)

In [437]:
family_importance = (
    coef_df
    .groupby("Family")["Abs_Coefficient"]
    .sum()
    .sort_values(ascending=False)
)

family_importance

Family
MA_Distance    4.272804
Momentum       4.011598
Volatility     3.984563
Mean_Return    3.222747
Name: Abs_Coefficient, dtype: float64

In [438]:
print(accuracy_score(y_val,val_pred))

0.5454545454545454


In [439]:
print(classification_report(y_val,val_pred))

              precision    recall  f1-score   support

         0.0       0.72      0.55      0.62       166
         1.0       0.35      0.54      0.43        76

    accuracy                           0.55       242
   macro avg       0.54      0.54      0.53       242
weighted avg       0.61      0.55      0.56       242



In [440]:
y_train.value_counts()

Target
0.0    1149
1.0     547
Name: count, dtype: int64

In [441]:
from xgboost import XGBClassifier

xgb = XGBClassifier(max_depth = 3,
learning_rate = 0.05,
n_estimators = 150,
scale_pos_weight = y_train.value_counts()[0]/y_train.value_counts()[1]
)

In [442]:
xgb.fit(x_train,y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [443]:
y_pred = xgb.predict(x_val)

In [444]:
print(classification_report(y_val,y_pred))

              precision    recall  f1-score   support

         0.0       0.69      0.70      0.69       166
         1.0       0.32      0.32      0.32        76

    accuracy                           0.58       242
   macro avg       0.51      0.51      0.51       242
weighted avg       0.58      0.58      0.58       242



In [445]:
# from sklearn.model_selection import GridSearchCV , TimeSeriesSplit

# model = XGBClassifier(scale_pos_weight = y_train.value_counts()[0]/y_train.value_counts()[1],random_state=42)

# tscv = TimeSeriesSplit(n_splits=5)

# param_grid = {
#     "max_depth": [1,2, 3, 4,5],
#     "learning_rate": [0.03, 0.05, 0.1,0.01,0.008,0.0075,0.007],
#     "n_estimators": [10,15,25,50,100]
# }

# # param_grid = {
# #     "max_depth": [2, 3, 4],
# #     "learning_rate": [0.005, 0.008, 0.01, 0.015],
# #     "n_estimators": [10, 20, 30, 50]
# # }

# grid = GridSearchCV(
#     estimator = model,
#     param_grid=param_grid,
#     scoring = 'recall',
#     cv=tscv,
#     n_jobs = -1,
#     verbose = 2
# )

# grid.fit(x_train,y_train)

In [446]:
# print(grid.best_params_)
# print(grid.best_score_)

In [447]:
xgb = XGBClassifier(max_depth = 5,
learning_rate = 0.008,
n_estimators = 10,
scale_pos_weight = y_train.value_counts()[0]/y_train.value_counts()[1]
)

In [448]:
xgb.fit(x_train,y_train)

y_pred = xgb.predict(x_val)

print(classification_report(y_val,y_pred))

              precision    recall  f1-score   support

         0.0       0.75      0.37      0.50       166
         1.0       0.35      0.72      0.47        76

    accuracy                           0.48       242
   macro avg       0.55      0.55      0.48       242
weighted avg       0.62      0.48      0.49       242



In [449]:
z = xgb.feature_importances_

In [450]:
w = list(x_train.columns)

In [451]:
w[1]

'ONGC.NS_Close_mom_3'

In [452]:
z[0]

np.float32(0.015274528)

In [453]:
coeff_dict = {}
for i in range(0,x_train.shape[1]):
    coeff_dict[w[i]] = (z[i])

In [454]:
coef_df = pd.DataFrame(
    coeff_dict.items(),
    columns=["Feature", "Coefficient"]
)

In [455]:
coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()

In [456]:
coef_df = coef_df.sort_values(
    by="Abs_Coefficient",
    ascending=False
)

In [457]:
coef_df["Ticker"] = coef_df["Feature"].str.split("_").str[0]

In [458]:
coef_df.groupby("Ticker")["Abs_Coefficient"].sum().sort_values(ascending=False)

Ticker
RELIANCE.NS    0.316361
BPCL.NS        0.280980
ONGC.NS        0.249161
^NSEI          0.153498
Name: Abs_Coefficient, dtype: float32

In [459]:
coef_df["Family"] = coef_df["Feature"].apply(get_family)

In [460]:
family_importance = (
    coef_df
    .groupby("Family")["Abs_Coefficient"]
    .sum()
    .sort_values(ascending=False)
)

family_importance

Family
Volatility     0.392535
Momentum       0.296018
MA_Distance    0.185002
Mean_Return    0.126445
Name: Abs_Coefficient, dtype: float32

In [461]:
x_train_final = pd.concat([x_train, x_val])
y_train_final = pd.concat([y_train, y_val])

In [462]:
final_xgb =XGBClassifier(max_depth = 5,
learning_rate = 0.008,
n_estimators = 10,
scale_pos_weight = y_train_final.value_counts()[0]/y_train_final.value_counts()[1]
)

In [463]:
final_xgb.fit(x_train_final, y_train_final)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [464]:
test_pred = final_xgb.predict(x_test)

In [465]:
print(classification_report(y_test, test_pred))

              precision    recall  f1-score   support

         0.0       0.75      0.58      0.66       374
         1.0       0.20      0.36      0.26       111

    accuracy                           0.53       485
   macro avg       0.48      0.47      0.46       485
weighted avg       0.63      0.53      0.57       485

